# Lezione 3. Processi Decisionali di Markov (PDM)

**Durata stimata:** 105 minuti
**Capitolo del manuale:** [04_procesos_decision_markov.md](../manual/04_procesos_decision_markov.md)

## Dalla slot machine alla mappa completa

Nella lezione precedente hai conosciuto i bandit: un agente che sceglie tra diverse leve e deve solo bilanciare esplorazione e sfruttamento, perché ogni tiro è indipendente da quelli precedenti. Oggi compi un passo in più.

La vita reale quasi mai funziona senza memoria. Quando guidi, annaffi una coltura o gestisci l'inventario di un magazzino, ogni decisione ti lascia in una situazione diversa che condiziona quali opzioni hanno senso dopo. Serve un linguaggio formale che catturi questa catena di causa ed effetto: il **Processo Decisionale di Markov** (PDM), in inglese *Markov Decision Process* (MDP) — troverai entrambi i termini nel codice e nella letteratura —.

Alla fine della lezione avrai in Python un mondo a griglia (GridWorld) funzionante, che la prossima lezione amplierà per trovare la strategia migliore possibile.

### Obiettivi di oggi

- Identificare i cinque elementi di un PDM: S, A, P, R e γ
- Capire la proprietà di Markov e perché semplifica il problema
- Distinguere una politica deterministica da una stocastica
- Calcolare Vπ di una politica fissa tramite valutazione iterativa

> 🎙️ **Traccia della lezione:** apri chiedendo "quale decisione avete preso oggi che dipendeva da una decisione precedente?". Lascia rispondere due o tre persone prima di introdurre il PDM: serve a ancorare l'idea che lo stato conta.


## 1. I cinque elementi di un PDM

Pensa a un robot che raccoglie lattine di alluminio per il riciclo e funziona a batteria. In ogni istante si trova in uno dei due **stati** possibili in base al suo livello di carica: alto o basso. A seconda dello stato in cui si trova, può scegliere tra diverse **azioni**: aspettare fermo, andare a cercare lattine o, se è a corto di energia, tornare alla base per ricaricarsi.

Ecco i cinque ingredienti del PDM:

- **S** (states, stati): l'insieme delle situazioni possibili in cui può trovarsi l'agente — per il robot, {carica alta, carica bassa} —
- **A** (actions, azioni): le opzioni disponibili in ogni stato — aspettare, cercare o ricaricare —
- **P**: la funzione di transizione, cioè la probabilità di finire in ciascuno stato successivo in base a dove eri e all'azione che hai scelto. Cercare con la batteria alta quasi sempre ti lascia con la stessa carica, ma esiste una probabilità di restare senza energia
- **R**: la funzione di ricompensa, il numero che ricevi dopo ogni transizione. Cercare lattine dà una ricompensa positiva; restare bloccato senza batteria, una forte penalità
- **γ** (gamma): il fattore di sconto, un numero tra zero e uno che decide quanto ti importa del futuro

Un PDM completo è la tupla **(S, A, P, R, γ)**.

> 🎙️ **Traccia della lezione:** disegna alla lavagna i due stati del robot e le frecce di transizione mentre lo spieghi. È l'immagine mentale che riutilizzeranno per tutta la lezione.


## 2. La proprietà di Markov: conta solo il presente

Il nome "Markov" non è decorativo. Il PDM assume la **proprietà di Markov**: il futuro dipende unicamente dallo stato attuale, non da come ci si è arrivati. Non importa se il robot ha passato due ore a cercare lattine o se si è appena ricaricato: l'unica cosa rilevante per decidere cosa fare adesso è il suo livello di batteria in questo momento.

Questa ipotesi semplifica enormemente il problema: non serve conservare tutto lo storico delle decisioni passate, basta uno stato ben progettato che riassuma ciò che conta. Per questo, quando progetti un sistema reale — per esempio, lo stato di un veicolo autonomo, che potrebbe includere posizione, velocità e ostacoli vicini —, il tuo compito è scegliere una rappresentazione che si avvicini il più possibile a questa proprietà.

## 3. La politica: la tua strategia di decisione

Una **politica**, scritta come π (pi), è la strategia seguita dall'agente: una regola che, per ogni stato, indica quale azione intraprendere. Può essere:

- **Deterministica**: nello stato "batteria bassa", π sceglie sempre "ricaricare"
- **Stocastica**: nello stato "batteria alta", π sceglie "cercare" il 70% delle volte e "aspettare" il restante 30%

Una politica casuale, che distribuisce la stessa probabilità tra tutte le azioni disponibili, è il punto di partenza più semplice per ragionare su un PDM, ed è proprio quella che programmerai oggi.

> 🎙️ **Traccia della lezione:** chiedi alla classe se una politica stocastica sembra "peggiore" di una deterministica. Approfitta per anticipare che nell'RL a volte conviene mantenere un po' di casualità (collegamento con l'esplorazione della lezione sui bandit).


## 4. Quanto vale uno stato: la funzione di valore

Ecco la domanda centrale di oggi: se segui una politica π, quanto "vale" trovarsi in un determinato stato? La risposta non è solo ciò che guadagni in questo istante, ma la somma di tutte le ricompense future, scontate di γ. Questa somma si chiama **ritorno** (Gt):

```
Gt = Rt+1 + γ·Rt+2 + γ²·Rt+3 + ...
```

Il fattore γ fa due cose insieme: riflette il fatto che di solito preferisci guadagnare qualcosa oggi piuttosto che la stessa ricompensa tra dieci passi, e impedisce che la somma cresca senza limiti nei compiti che potrebbero continuare indefinitamente, perché ogni ricompensa futura pesa un po' meno della precedente. Un γ vicino a zero produce un agente miope, che guarda solo al passo immediato; uno vicino a uno lo rende lungimirante, disposto a sacrificare una ricompensa immediata per un beneficio maggiore più avanti.

A partire dal ritorno si definiscono due **funzioni di valore**:

- **Vπ(s)**: il ritorno atteso se parti dallo stato s e segui la politica π da lì in avanti
- **Qπ(s,a)**: il ritorno atteso se, trovandoti nello stato s, prendi l'azione a e poi segui π

Le due sono legate in modo diretto: Vπ(s) è la media di Qπ(s,a) su tutte le azioni, ponderata per la probabilità che la politica assegna a ciascuna.


## 5. Le equazioni di Bellman

Il ritorno si può scrivere in forma ricorsiva, perché Gt = Rt+1 + γ·Gt+1. In parole: il valore di questo momento è la ricompensa di questo passo più il valore scontato di tutto ciò che viene dopo. Questa idea, applicata alla funzione di valore, è la **equazione di Bellman di aspettazione**:

```
Vπ(s) = Σ_a π(a|s) · Σ_s' P(s'|s,a) · [R(s,a,s') + γ·Vπ(s')]
```

Non lasciarti spaventare dalla notazione: è solo una somma ponderata. Per ogni azione possibile (secondo la tua politica) e ogni possibile stato successivo (secondo la probabilità di transizione), sommi la ricompensa immediata più il suo valore scontato. La stessa idea, applicata a Q, dà la versione per le coppie stato-azione:

```
Qπ(s,a) = Σ_s' P(s'|s,a) · [R(s,a,s') + γ·Σ_a' π(a'|s') Qπ(s',a')]
```

Queste due equazioni definiscono il valore di una politica già fissata: sono un sistema lineare, e per questo puoi risolverlo — oppure approssimarlo iterando, come farai nel codice di oggi —.

> 🎙️ **Traccia della lezione:** insisti sulla lettura a voce della formula ("ricompensa immediata + valore scontato dello stato successivo") prima di toccare la tastiera. È la frase che ripeteranno tra poco in ogni riga di codice.

### Anticipazione: la domanda che apre la prossima lezione

Tutto ciò che precede presuppone che tu abbia già una politica π e voglia solo sapere quanto vale. Ma la domanda che davvero interessa è un'altra: qual è la migliore politica possibile? Questa politica ottimale si chiama **π\***, e la sua funzione di valore **V\*(s)** soddisfa la **equazione di ottimalità di Bellman**:

```
V*(s) = max_a Σ_s' P(s'|s,a) · [R(s,a,s') + γ·V*(s')]
```

La differenza rispetto all'equazione di aspettazione è sottile ma decisiva: invece di fare la media sulle azioni secondo la tua politica, prendi il massimo. Quel massimo rende il sistema non lineare e richiede metodi iterativi, che è proprio ciò di cui tratta la Lezione 4: la programmazione dinamica. Oggi ti fermi a metà strada: saprai misurare una politica, ma non ancora migliorarla.


## 6. Manos a la obra: il GridWorld

Costruirai un piccolo mondo a griglia, di quattro righe per quattro colonne (4×4), con un'unica casella terminale (l'obiettivo) nell'angolo in basso a destra. L'agente può muoversi nelle quattro direzioni cardinali; se cerca di uscire dal tabellone, resta dov'è. Ogni passo costa −1 di ricompensa, tranne l'arrivo all'obiettivo, dove l'episodio termina.

Definirai una politica casuale e calcolerai il suo Vπ applicando più volte l'equazione di Bellman di aspettazione, finché i valori smettono di cambiare. Questo processo si chiama **valutazione iterativa di politiche**.

Nota che la classe `GridWorld` e le funzioni sono scritte per essere riutilizzate: la Lezione 4 aggiungerà il miglioramento delle politiche su questa stessa base, senza riscriverla. Costruisci il codice cella per cella, eseguendo ciascuna prima di passare alla successiva.

> 🎙️ **Traccia della lezione:** prima di scrivere codice, disegna la griglia 4x4 alla lavagna con l'obiettivo segnato. Numera un paio di caselle perché la classe visualizzi le coordinate (riga, colonna) prima di vederle in Python.


In [ ]:
# Solo necesitas numpy, ya viene preinstalado en Google Colab
import numpy as np

### Le azioni e i loro spostamenti

Ogni azione sposta l'agente di una casella in una direzione. Lo rappresentiamo come un delta di riga e un delta di colonna.


In [ ]:
# 🎙️ Guion: pregunta "¿por qué represento 'arriba' como (-1, 0) y no como (0, -1)?"
# Respuesta: la primera coordenada es la fila, y subir una fila resta uno al índice.
ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

### La classe `GridWorld`

Questa classe incapsula l'ambiente completo: quali stati esistono, come si muove l'agente e quale ricompensa riceve. È progettata perché la Lezione 4 la riutilizzi così com'è e vi aggiunga iterazione delle politiche e iterazione dei valori, quindi mantieni esattamente i suoi nomi se la modifichi in seguito.


In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta).

    - Los estados son tuplas (fila, columna).
    - Las acciones son las cuatro direcciones cardinales.
    - Las transiciones son deterministas: moverse contra un muro o el
      borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None, recompensa_paso=-1.0):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        # Por defecto, la meta es la esquina inferior derecha
        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)

        # Lista de todos los estados (fila, columna)
        self.estados = [(f, c) for f in range(n_filas) for c in range(n_columnas)]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero, el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        return (nueva_fila, nueva_columna)

    def recompensa(self, estado, accion, estado_siguiente):
        """Recompensa inmediata de la transición. En este GridWorld es fija
        (-1 por paso), pero se recibe (s, a, s') completo para poder
        ampliar el ejemplo más adelante (por ejemplo, recompensas distintas por casilla)."""
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso

### Verifica rapida

Prima di continuare, crea un `GridWorld` e verifica che si comporti come previsto: che l'angolo in basso a destra sia terminale e che muoversi contro il bordo ti lasci nello stesso punto.

> 🎙️ **Traccia della lezione:** chiedi a qualcuno della classe di prevedere ad alta voce il risultato di `estado_siguiente((0, 0), "arriba")` prima di eseguire la cella.


In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)

print("¿(3, 3) es terminal?", mundo.es_terminal((3, 3)))
print("Moverse 'arriba' desde (0, 0):", mundo.estado_siguiente((0, 0), "arriba"))
print("Moverse 'derecha' desde (1, 1):", mundo.estado_siguiente((1, 1), "derecha"))

### La politica casuale

Rappresentiamo una politica come un dizionario annidato: `politica[estado][accion]` è la probabilità di prendere quell'azione in quello stato. La politica casuale distribuisce la stessa probabilità tra le quattro direzioni.


In [ ]:
def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25).
    Se representa como un diccionario: politica[estado][accion] = probabilidad."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }

politica = politica_aleatoria(mundo)

# Inspecciona la política en un único estado
print("Política en el estado (2, 2):", politica[(2, 2)])

### Valutazione iterativa di politiche

Qui sta il cuore della lezione di oggi: applicare ripetutamente l'equazione di Bellman di aspettazione, passata dopo passata, finché Vπ smette di cambiare (sotto una `umbral`, soglia).

Nota la corrispondenza riga per riga con la formula: `prob_accion * (r + gamma * V[estado_sig])` è esattamente `π(a|s) · [R(s,a,s') + γ·Vπ(s')]`. Poiché questo GridWorld è deterministico, P(s'|s,a) vale uno per l'unico `estado_siguiente` restituito da `mundo.estado_siguiente()`, quindi la somma su s' scompare e resta un unico termine.

> 🎙️ **Traccia della lezione:** prima di eseguire, indica alla lavagna quale riga di codice corrisponde a ciascun simbolo della formula (π(a|s), R(s,a,s'), γ·Vπ(s')). È il momento di maggiore densità concettuale della lezione: prenditi il tuo tempo.


In [ ]:
def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """
    Calcula Vpi(s) para una política fija aplicando repetidamente
    la ecuación de Bellman de expectación:

        Vpi(s) = suma_a pi(a|s) * suma_s' P(s'|s,a) * [R(s,a,s') + gamma * Vpi(s')]

    Devuelve V como un diccionario {estado: valor} y el número de barridos usados.
    """
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)  # barrido síncrono: usamos siempre los valores del barrido anterior

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion

### Visualizzare il risultato

Una funzione ausiliaria per stampare V come una griglia leggibile, invece che come un dizionario piatto.


In [ ]:
def imprimir_valores(mundo, V):
    """Muestra V como una cuadrícula legible, para inspeccionar el resultado."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            fila_textos.append(f"{V[(f, c)]:6.2f}")
        print(" ".join(fila_textos))

### Esegui la valutazione completa

Metti insieme tutti i pezzi: crea il mondo, la politica casuale, valuta Vπ e stampa il risultato.

> 🎙️ **Traccia della lezione:** prima di eseguire, chiedi "in quale casella vi aspettate il valore più alto (meno negativo)?". Verifica l'intuizione con il risultato.


In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
politica = politica_aleatoria(mundo)

V, num_barridos = evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4)

print(f"Convergió en {num_barridos} barridos.\n")
print("Función de valor Vpi bajo la política aleatoria:")
imprimir_valores(mundo, V)

### Come leggere il risultato

Dovresti ottenere qualcosa di simile a questo (i valori esatti dipendono da γ e dalla soglia di convergenza):

```
Convergenza raggiunta in 76 iterazioni.

Funzione di valore Vpi con la politica casuale:
 -9.36  -9.22  -8.97  -8.75
 -9.22  -8.97  -8.50  -7.97
 -8.97  -8.50  -7.42  -5.76
 -8.75  -7.97  -5.76   0.00
```

Osserva lo schema: le caselle più vicine all'obiettivo (l'angolo in basso a destra, con 0.00) hanno un valore meno negativo, perché sotto una politica casuale servono in media meno passi per arrivarci. Quelle più lontane, come l'angolo in alto a sinistra, accumulano più passi di ricompensa −1 prima di terminare. È esattamente ciò che misura Vπ: il costo atteso, scontato, di arrivare alla fine seguendo la tua politica attuale.

> 🎙️ **Traccia della lezione:** lascia che la classe confronti lo schema di numeri con il disegno alla lavagna. Chiedi: "perché l'angolo in alto a sinistra è la casella peggiore possibile qui?".


## Punti chiave

- Un PDM è definito dalla tupla (S, A, P, R, γ): stati, azioni, transizioni, ricompense e fattore di sconto
- La proprietà di Markov afferma che il futuro dipende solo dallo stato presente, non dallo storico completo
- Una politica π può essere deterministica o stocastica, e assegna a ogni stato un'azione o una distribuzione di probabilità su di esse
- Vπ(s) misura il ritorno atteso partendo da uno stato e seguendo π; Qπ(s,a) fa lo stesso ma fissando anche la prima azione
- L'equazione di Bellman di aspettazione scompone il valore attuale in ricompensa immediata più il valore scontato dello stato successivo
- L'equazione di ottimalità di Bellman sostituisce la media ponderata con un massimo sulle azioni, e definisce V* e la politica ottimale π*
- La valutazione iterativa di politiche applica ripetutamente l'equazione di Bellman di aspettazione finché Vπ converge

## Chiusura e prossimi passi

Hai la mappa — il PDM completo — e sai calcolare quanto vale seguire una politica concreta. Nella Lezione 4 userai questo stesso `GridWorld` per fare il passo successivo: invece di valutare soltanto una strategia fissa, la migliorerai passo dopo passo tramite iterazione delle politiche e iterazione dei valori, fino a raggiungere la politica ottimale π*.

**Prima della prossima lezione:**
- **Apri** il notebook di pratica `clase03_practica_procesos_decision_markov.ipynb` e completa i due esercizi
- **Sperimenta** cambiando γ a 0.5 e a 0.99: annota come cambiano i valori e il numero di iterazioni necessarie per convergere
- **Ripassa** il capitolo 4 del manuale se vuoi approfondire la derivazione matematica

> 🎙️ **Traccia della lezione:** chiudi ricordando la frase chiave di oggi: "ricompensa immediata + valore scontato dello stato successivo". È la stessa frase che ripeteranno la prossima lezione, ora con un massimo al posto di una media.
